# 07 — The McCann Geodesic

Close the Wasserstein arc with its most beautiful feature: the $W_2$ geodesic *slides* mass through the ground space — the transport-geometry half of the two-geometries hinge.

**Prerequisites:** `05_wasserstein_distance`, `06_1d_quantile_closed_form`, `02/08_two_geometries_one_simplex`.

**What you'll be able to do**
- Compute the McCann displacement geodesic $\mu_t$ between two distributions at any time $t$.
- Contrast it with the mixture path — sliding mass versus morphing amplitudes.
- Place this beside the two-geometries result of Module 2 and read the transport row of the dictionary.

You have a Wasserstein distance (05) and a fast way to compute it on the line (06). This synthesis adds the last piece: the *geodesic* — the straight line of the transport geometry. Where the information geometry of Module 2 interpolated by fading one bump up and another down, transport slides a single bump bodily across the space. Seeing those two midpoints side by side is the heart of why optimal transport is a different kind of geometry.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.geometry.info_geometry import mixture_interpolation
from qot_course.transport.wasserstein import mccann_geodesic_atoms_1d

np.random.seed(42)
viz.use_course_style()

## Where we are

Two bricks brought us here: `05_wasserstein_distance` made $W_p$ a genuine metric that respects ground geometry, and `06_1d_quantile_closed_form` gave the fast sort-and-match computation on the line. Now we ask what the *straight lines* of this geometry look like — and collect the transport row the arc has earned.

## 1. McCann displacement interpolation — the $W_2$ geodesic

The Wasserstein-2 distance is more than a metric: $(\mathcal{P}_2, W_2)$ is a geodesic space (Otto, 2001). The geodesic between $\mu$ and $\nu$ is the **McCann interpolation**

$$ \mu_t = \bigl( (1 - t)\,\mathrm{Id} + t\,T \bigr)_{\#}\,\mu, \qquad t \in [0, 1], $$

where $T = F_\nu^{-1} \circ F_\mu$ is the optimal map from `06`. Each unit of mass slides *linearly* from its $\mu$-position to its $\nu$-position. We compute the geodesic at a few times and, alongside it, the **mixture path** $(1-t)\mu + t\nu$ — the information-geometry interpolation from Module 2 — to see how differently they move.

In [ ]:
grid = np.linspace(-1.0, 11.0, 600)
dx = grid[1] - grid[0]

def density(center, sigma):
    """Gaussian bump on the grid, normalised to a probability density."""
    pdf = np.exp(-0.5 * ((grid - center) / sigma) ** 2)
    return pdf / np.trapezoid(pdf, grid)

mass_mu = density(3.0, 0.8) * dx; mass_mu /= mass_mu.sum()
mass_nu = density(7.0, 1.2) * dx; mass_nu /= mass_nu.sum()

times = [0.0, 0.25, 0.5, 0.75, 1.0]
mccann_snapshots = [mccann_geodesic_atoms_1d(grid, mass_mu, grid, mass_nu, t) for t in times]
mixture_snapshots = [mixture_interpolation(mass_mu, mass_nu, t) for t in times]
print(f"computed {len(times)} snapshots for both the McCann geodesic and the mixture path")

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(10, 7), sharex=True)
cmap = plt.colormaps["viridis"]
edges = np.concatenate([[grid[0] - dx / 2], 0.5 * (grid[:-1] + grid[1:]), [grid[-1] + dx / 2]])

for k, ((positions_t, weights_t), t) in enumerate(zip(mccann_snapshots, times)):
    counts, _ = np.histogram(positions_t, bins=edges, weights=weights_t)
    axes[0].plot(grid, counts, color=cmap(k / (len(times) - 1)), lw=2, label=f"t={t:.2f}")
axes[0].set_ylabel("mass")
axes[0].set_title("McCann geodesic (W2): the single peak slides across", pad=10)
axes[0].legend(loc="upper right", ncol=5, fontsize=9)

for k, (pt, t) in enumerate(zip(mixture_snapshots, times)):
    axes[1].plot(grid, pt, color=cmap(k / (len(times) - 1)), lw=2, label=f"t={t:.2f}")
axes[1].set_xlabel("position  x")
axes[1].set_ylabel("mass")
axes[1].set_title("Mixture path (information side): two fixed peaks, bimodal at t=0.5", pad=10)
axes[1].legend(loc="upper right", ncol=5, fontsize=9)
plt.tight_layout()
plt.show()

**Read both panels.** Top — the $W_2$ McCann geodesic: a single bump *slides* smoothly from $x = 3$ to $x = 7$, staying unimodal the whole way. Transport sees that the bins live on a line with empty space between the bumps, and chooses to carry the mass *through* that space. Bottom — the mixture path: two fixed peaks whose heights trade off, plainly **bimodal** at $t = 0.5$. This is exactly the two-geometries punchline of `02/08_two_geometries_one_simplex`, now backed by the formal $W_2$ geodesic: the information geometry morphs amplitudes in place, while the transport geometry slides mass across the ground space.

## 2. Dictionary update — the transport row, formalised

The arc adds three transport entries to the running classical↔quantum dictionary: the Wasserstein distance itself, its one-dimensional closed form (which the Gaussian case lifts to the Bures–Wasserstein formula), and the McCann geodesic (whose quantum analogue is displacement interpolation on states).

| Classical | Quantum |
|-----------|---------|
| probability vector $a$ on positions $\{x_i\}$ | density matrix $\rho$ |
| KL divergence $D(p\|q)$ | Umegaki relative entropy $S(\rho\|\sigma)$ |
| transport plan / coupling $P$ | quantum coupling: bipartite $\rho_{AB}$  *(Module 4)* |
| Kantorovich LP $\min \langle C, P\rangle$ | quantum OT SDP $\min \mathrm{tr}(C\,\rho_{AB})$  *(Module 4)* |
| **Wasserstein-$p$ distance $W_p(\mu, \nu)$** | **quantum Wasserstein**  *(Module 4)* |
| **1-D closed form $W_p^p = \int |F_\mu^{-1} - F_\nu^{-1}|^p$** | **Gaussian closed form (Bures–Wasserstein, `12`)** |
| **McCann geodesic $\mu_t$** | **quantum displacement interpolation**  *(Module 4)* |

The transport geometry is now a place with both distances and straight lines.

## Your turn

1. **A bimodal challenge.** Take $\mu = \tfrac{1}{2}\delta_{-1} + \tfrac{1}{2}\delta_{+1}$ and $\nu = \delta_0$. Compute the McCann midpoint $\mu_{1/2}$. Is it bimodal, unimodal, or a single atom — and what does that say about how transport handles split mass?
2. **Concentration at the midpoint.** Measure how spread out the McCann midpoint is versus the mixture midpoint (compare their variances). Which path keeps the mass more concentrated, and why?
3. **Constant speed.** A geodesic has constant speed: $W_2(\mu, \mu_t)$ should equal $t \cdot W_2(\mu, \nu)$. Check this at a few values of $t$ using `mccann_geodesic_atoms_1d` and your distance from `06`.

## What you built

- You computed the McCann $W_2$ geodesic and watched a single bump slide from source to target.
- You contrasted it with the mixture path, which stays bimodal — the two geometries of Module 2 made concrete and quantitative.
- You formalised the transport row of the classical↔quantum dictionary.

That closes the Wasserstein arc: the transport geometry is now a full geometry, with distances *and* straight lines. You can measure how far two distributions are and trace the optimal path between them — the exact structure we will lift to quantum states.

## What's next

The LP gives a distance; computing it in high dimensions is still costly. In `08_kantorovich_duality` we open the next arc by looking at the LP from its *dual* side — potentials instead of plans — which leads through entropic regularisation and the Sinkhorn algorithm to Amari's bridge, the place where Wasserstein meets the KL divergence of Module 2.

## References

- R. McCann, "A convexity principle for interacting gases", *Advances in Mathematics* 128, 153–179 (1997). DOI:10.1006/aima.1997.1634
- F. Otto, "The geometry of dissipative evolution equations: the porous medium equation", *Comm. PDE* 26, 101–174 (2001).
- C. Villani, *Topics in Optimal Transportation*, AMS (2003), ch. 5.
- G. Peyré & M. Cuturi, *Computational Optimal Transport*, NoW (2019), ch. 7. DOI:10.1561/2200000073

**Previous:** `notebooks/03_ClassicalOptimalTransport/06_1d_quantile_closed_form.ipynb` · **Next:** `notebooks/03_ClassicalOptimalTransport/08_kantorovich_duality.ipynb`